# InfoQuant-02 — Trois copies de Werner : la frontière de distillabilité

**Deuxième carnet de la sous-série** [Information quantique](README.md) — le prolongement direct du §5 d'[InfoQuant-01](InfoQuant-01-Intrication-Werner-PPT-Distillation-Python.ipynb), qui posait le vocabulaire (états de Werner, critère PPT, distillabilité) et terminait sur un trou : *entre* l'état séparable et l'état distillable à une copie, que se passe-t-il ?

Ce carnet digère un résultat de recherche récent qui avance dans ce trou : *Three copies of the Werner state: the sharp curve in dimensions three and four and a bound in every dimension* (Ian MacCormack, Physical Superintelligence PBC ; Ning Bao, Northeastern/BNL ; preprint du 23 octobre 2026, 14 pages), publié dans le lot [`oqp-batch-2026`](https://github.com/physical-superintelligence/oqp-batch-2026) comme réponse partielle au problème ouvert n° 2 de l'IQOQI Vienne : *Undistillability implies PPT?* Le résultat tient en trois phrases :

1. **La courbe exacte** (dimensions locales 3 et 4) : le minimum de la forme à trois copies $q_\alpha$ sur les matrices unité de rang au plus deux vaut exactement $(1-2\alpha)(1-\alpha)^2$ — donc trois copies de l'état de Werner $\rho_\alpha$ sont distillables **si et seulement si** $\alpha > 1/2$.
2. **Une borne en toute dimension** : trois copies ne suffisent pas pour tout $\alpha \leq 37/80 = 0{,}4625$, en toute dimension locale finie — donc des états de Werner NPT qui survivent à trois copies existent pour toute dimension $d \geq 3$.
3. **Ce qui reste ouvert** : en dimension $d \geq 5$, le comportement à la borne $\alpha = 1/2$ n'est pas tranché.

Le carnet suit la convention de la sous-série : chaque seuil cité est d'abord **retrouvé numériquement**, puis confronté à l'énoncé de la source. Ce que ce carnet **ne fait pas** : il ne redémontre ni l'inégalité d'extrémité (certificats rationnels computer-vérifiés, §4) ni la borne uniforme (§5) ; il les met en marche sur des matrices explicites.

> **Statut de la source.** Preprint non relu. Les auteurs déclarent la provenance : mathématiques développées par les chercheurs de PSI avec des modèles d'IA en flux de recherche automatisés, formalisation Lean par un agent ; MacCormack et Bao sont responsables de la correction. Le classement du problème comme **ouvert** vient du [catalogue OQP](https://oqp.iqoqi.oeaw.ac.at/undistillability-implies-ppt) de l'IQOQI, cité tel quel — un preprint ne bascule pas un classement.

## 0. Le problème : des états intriqués impossibles à distiller ?

La **distillation** (Bennett et al. 1996, §5 d'InfoQuant-01) extrait des paires quasi parfaites de nombreuses copies d'un état bruité. Un état PPT n'est jamais distillable. La question du catalogue OQP n° 2 s'écrit dans l'autre sens : *si un état n'est pas distillable — même avec un nombre arbitraire de copies — est-il forcément PPT ?* L'alternative serait un **état intriqué lié** (*bound entangled*) à transposée partielle négative : intriqué, mais dont l'intrication ne se laisse jamais concentrer.

Les états de Werner $\rho_\alpha$ sont le terrain d'essai le plus simple. Ils vivent sur $\mathbb{C}^d \otimes \mathbb{C}^d$ et forment une famille à un paramètre :

- $\rho_\alpha$ est **NPT** (transposée partielle non positive) dès que $\alpha > 1/d$ ;
- $\rho_\alpha$ est **distillable à une copie** dès que $\alpha > 1/2$ (seuil de Dür et al. 2000, déjà rencontré en §5 d'InfoQuant-01) ;
- entre les deux — la bande $1/d < \alpha \leq 1/2$ — le signe de l'intrication est la question ouverte.

Le résultat digéré ici mesure cette bande **à trois copies** : en dimensions 3 et 4, trois copies ne suffisent jamais dans la bande (le seuil de distillabilité à trois copies est exactement $\alpha = 1/2$, le même qu'à une copie) ; en toute dimension, une portion $\alpha \leq 37/80$ de la bande résiste aussi à trois copies.

Le carnet procède en six temps : la paramétrisation du papier et son pont avec le $\lambda$ d'InfoQuant-01 (§1), le critère de distillabilité réécrit comme une forme sur les matrices — le moteur numérique du carnet (§2), la courbe exacte et son minimiseur (§3), ce que l'ordinateur vérifie en amont (§4), la borne uniforme $37/80$ et la frontière ouverte en grande dimension (§5), la carte du trou et ce que le résultat dit — et ne dit pas — du problème de l'IQOQI (§6).

In [1]:
import numpy as np

rng = np.random.default_rng(42)
print("numpy", np.__version__)


def ket(*etats):
    """Produit tensoriel de kets locaux (vecteurs numpy 1-D). (InfoQuant-01, section 1)"""
    v = np.array([1.0])
    for e in etats:
        v = np.kron(v, np.asarray(e, dtype=complex))
    return v


def val_propres(rho):
    """Valeurs propres reelles d'une matrice hermitienne, triees croissant."""
    return np.sort(np.real(np.linalg.eigvalsh(rho)))


def tp_b(rho, d1, d2):
    """Transpose partielle sur le facteur B (convention de Peres, InfoQuant-01 section 4)."""
    R = rho.reshape(d1, d2, d1, d2)
    return R.transpose(0, 3, 2, 1).reshape(d1 * d2, d1 * d2)


def flip(d):
    """Operateur de retournement F|ij> = |ji> sur C^d (x) C^d (InfoQuant-01, section 3)."""
    V = np.zeros((d * d, d * d))
    for i in range(d):
        for j in range(d):
            V[i * d + j, j * d + i] = 1.0
    return V


print("Organes herites d'InfoQuant-01 charges : ket, val_propres, tp_b, flip")

numpy 2.5.2
Organes herites d'InfoQuant-01 charges : ket, val_propres, tp_b, flip


## 1. La paramétrisation du papier, et son pont avec InfoQuant-01

Le papier de MacCormack et Bao (section II) écrit l'état de Werner

$$\rho_\alpha = \frac{I - \alpha F}{d(d-\alpha)},$$

où $F$ est l'opérateur de retournement. C'est **la même famille** que le $W(\lambda)$ d'InfoQuant-01 (revue Horodecki et al. 2009, éq. 71, où $\lambda$ est le poids de la partie antisymétrique), dans une autre échelle : le poids antisymétrique de $\rho_\alpha$ vaut

$$\lambda(\alpha) = \frac{(1+\alpha)(d-1)}{2(d-\alpha)}, \qquad \text{donc} \qquad \alpha(\lambda) = \frac{2\lambda d - d + 1}{2\lambda + d - 1}.$$

Deux repères se correspondent exactement : le seuil de séparabilité $\lambda = 1/2$ (InfoQuant-01, §3) correspond à $\alpha = 1/d$ (PPT), et le seuil de distillabilité à une copie $\lambda > 3(d-1)/(4d-2)$ (Dür et al., §5 d'InfoQuant-01) correspond à $\alpha > 1/2$. Le papier le confirme : $\rho_\alpha$ est NPT exactement quand $\alpha > 1/d$, et one-copy distillable exactement quand $\alpha > 1/2$ (section II, *Werner states and distillability*).

Construisons $\rho_\alpha$ et vérifions les deux repères spectraux.

In [2]:
def werner_alpha(alpha, d):
    """Werner du papier (eq. 1 de MacCormack-Bao) : rho_alpha = (I - alpha F) / (d (d - alpha))."""
    F = flip(d)
    return (np.eye(d * d) - alpha * F) / (d * (d - alpha))


def lambda_de_alpha(alpha, d):
    """Poids antisymetrique de rho_alpha (echelle lambda d'InfoQuant-01, Horodecki et al. eq. 71)."""
    return (1 + alpha) * (d - 1) / (2 * (d - alpha))


def alpha_de_lambda(lmbda, d):
    """Reciproque de lambda_de_alpha : alpha(lambda) = (2*lambda*d - d + 1) / (2*lambda + d - 1)."""
    return (2 * lmbda * d - d + 1) / (2 * lmbda + d - 1)


d = 3
F = flip(d)
P_moins = (np.eye(d * d) - F) / 2  # projecteur antisymetrique

# (a) Trace et poids antisymetrique : la conversion lambda(alpha) est exacte
for alpha in [0.1, 0.2, 1 / d, 0.4]:
    rho = werner_alpha(alpha, d)
    poids_anti = np.trace(P_moins @ rho) / np.trace(rho)
    print(f"alpha={alpha:.4f}  tr(rho)={np.trace(rho).real:.12f}  "
          f"lambda mesure={poids_anti.real:.10f}  lambda formule={lambda_de_alpha(alpha, d):.10f}")

# (b) Le seuil NPT : rho_alpha est NPT exactement quand alpha > 1/d (papier, section II)
print(f"\nd = {d} : seuil NPT attendu alpha > 1/d = {1/d:.6f}")
print("alpha     vp(rho) min        vp(rho^Gamma) min    NPT ?")
for alpha in [0.20, 1 / d - 1e-9, 1 / d + 1e-9, 0.40, 0.50]:
    rho = werner_alpha(alpha, d)
    vp_gamma = val_propres(tp_b(rho, d, d))[0]
    print(f"{alpha:.9f}  {val_propres(rho)[0]:+.10f}   {vp_gamma:+.12f}   {'oui' if vp_gamma < 0 else 'non'}")

# (c) Pont avec le seuil de Duer et al. : lambda(1/2) = 3(d-1)/(4d-2)
print(f"\nlambda(1/2) = {lambda_de_alpha(0.5, d):.10f}   seuil 1-copy Duer 3(d-1)/(4d-2) = {3 * (d - 1) / (4 * d - 2):.10f}")
print("Les deux seuils d'InfoQuant-01 section 5 et du papier coincident a l'echelle pres.")

alpha=0.1000  tr(rho)=1.000000000000  lambda mesure=0.3793103448  lambda formule=0.3793103448
alpha=0.2000  tr(rho)=1.000000000000  lambda mesure=0.4285714286  lambda formule=0.4285714286
alpha=0.3333  tr(rho)=1.000000000000  lambda mesure=0.5000000000  lambda formule=0.5000000000
alpha=0.4000  tr(rho)=1.000000000000  lambda mesure=0.5384615385  lambda formule=0.5384615385

d = 3 : seuil NPT attendu alpha > 1/d = 0.333333
alpha     vp(rho) min        vp(rho^Gamma) min    NPT ?
0.200000000  +0.0952380952   +0.047619047619   non
0.333333332  +0.0833333334   +0.000000000375   non
0.333333334  +0.0833333332   -0.000000000375   oui
0.400000000  +0.0769230769   -0.025641025641   oui
0.500000000  +0.0666666667   -0.066666666667   oui

lambda(1/2) = 0.6000000000   seuil 1-copy Duer 3(d-1)/(4d-2) = 0.6000000000
Les deux seuils d'InfoQuant-01 section 5 et du papier coincident a l'echelle pres.


### Lecture du résultat

La conversion d'échelle est **mesurée exacte** : le poids antisymétrique calculé depuis la matrice $\rho_\alpha$ coïncide avec la formule $\lambda(\alpha) = (1+\alpha)(d-1)/(2(d-\alpha))$ à la précision flottante près, et le seuil NPT bascule exactement à $\alpha = 1/d$ (les deux valeurs propres minimales de part et d'autre du seuil, à $10^{-9}$ près, encadrent zéro). Le pont avec InfoQuant-01 est verrouillé : $\lambda(1/2) = 3(d-1)/(4d-2)$, c'est-à-dire que le seuil de distillabilité à une copie de Dür et al. (§5 d'InfoQuant-01) et le seuil $\alpha > 1/2$ du papier désignent le même état frontière.

À partir d'ici, le carnet parle en $\alpha$ — l'échelle du papier. Le lecteur peut retraduire en $\lambda$ par la formule ci-dessus.

## 2. La distillabilité comme une forme sur les matrices

C'est le cœur technique du papier (section II, *Distillability as a matrix inequality*) et le moteur numérique de tout ce carnet.

**Le critère.** Un état $\varrho$ est distillable à une copie s'il existe un vecteur $|\psi\rangle$ de rang de Schmidt au plus deux avec $\langle\psi, \varrho^\Gamma \psi\rangle < 0$, où $\Gamma$ est la transposition partielle. Pour $n$ copies, on applique le critère à $\varrho^{\otimes n}$.

**La forme à trois copies.** Puisque $(\rho_\alpha)^\Gamma = (I - \alpha\,\Omega)/d(d-\alpha)$, où $\Omega = d\,|\Phi^+\rangle\langle\Phi^+|$ (le facteur $d |\Phi^+\rangle\langle\Phi^+|$ agit comme l'identité croisée sur chaque site), le papier réécrit le critère sur une matrice $C$ de taille $d^3 \times d^3$ — vue comme un opérateur sur trois sites $H = H_1 \otimes H_2 \otimes H_3$, avec $\operatorname{vec} C$ le vecteur obtenu en empilant les colonnes. La **Proposition 2.1** du papier donne l'identité

$$q_\alpha(C) \;=\; \sum_{S \subseteq \{1,2,3\}} (-\alpha)^{|S|}\, \big\| \operatorname{Tr}_S C \big\|^2 \;=\; \Big\langle \operatorname{vec} C,\; \bigotimes_{i=1}^{3} (I - \alpha\, \Omega_i)\, \operatorname{vec} C \Big\rangle,$$

où $\operatorname{Tr}_S C$ est la trace partielle (non normalisée) sur les sites de $S$, et $\|\cdot\|$ la norme de Frobenius. Le dénominateur positif $d^3(d-\alpha)^3$ disparaît dans le signe, donc :

> $\rho_\alpha^{\otimes 3}$ est distillable à une copie — on dira **trois copies suffisent** — si et seulement s'il existe $C$ de rang au plus deux, de norme un, avec $q_\alpha(C) < 0$.

À $\alpha = 1/2$, la forme $q := q_{1/2} = \sum_S (-1/2)^{|S|} \|\operatorname{Tr}_S C\|^2$ est celle que les certificats computer-vérifiés contrôlent (§4). Les valeurs une-site servent de brique :

$$q^{(1)}_\alpha(A) = \|A\|^2 - \alpha\,|\operatorname{Tr} A|^2 \qquad \text{(le terme } S = \{i\} \text{ du produit tensoriel).}$$

Implémentons $q_\alpha$ par contractions d'indices — une matrice $d^3 \times d^3$ est un tenseur à six indices $(a_1 a_2 a_3 \,|\, b_1 b_2 b_3)$, et $\operatorname{Tr}_S$ contracte $a_i = b_i$ pour $i \in S$.

In [3]:
from itertools import combinations


def tr_sites(C, sites, d):
    """Trace partielle (non normalisee) de C sur les sites S d'un produit a 3 sites.

    C est une matrice d^3 x d^3, vue comme tenseur (a1 a2 a3 | b1 b2 b3) : les trois
    premiers axes sont les indices ligne, les trois suivants les indices colonne.
    Tracer le site s contracte a_s avec b_s. Le resultat vit sur les sites restants,
    sous forme d'une matrice carree (un scalaire pour S = {1,2,3}).
    """
    T = C.reshape(d, d, d, d, d, d)
    labels = ['a1', 'a2', 'a3', 'b1', 'b2', 'b3']
    for s in sorted(sites):
        ia = labels.index(f'a{s}')
        ib = labels.index(f'b{s}')
        T = np.trace(T, axis1=ia, axis2=ib)
        labels = [l for l in labels if l not in (f'a{s}', f'b{s}')]
    if T.ndim == 0:
        return np.array([[complex(T)]])
    side = int(round(T.size ** 0.5))
    return T.reshape(side, side)


def q_alpha(C, alpha, d):
    """Forme a trois copies du papier (Proposition 2.1) :

    q_alpha(C) = somme sur S sous-ensemble de {1,2,3} de (-alpha)^|S| * ||Tr_S C||^2_Frobenius.
    """
    total = 0.0
    for k in range(4):
        for S in combinations((1, 2, 3), k):
            TS = tr_sites(C, S, d)
            total += ((-alpha) ** k) * float(np.sum(np.abs(TS) ** 2))
    return total


def q_un_site(A, alpha):
    """Forme une-site : q^(1)_alpha(A) = ||A||^2 - alpha |Tr A|^2 (papier, section II)."""
    A = np.asarray(A)
    return float(np.sum(np.abs(A) ** 2)) - alpha * abs(np.trace(A)) ** 2


# --- Sanity 1 : la valeur sur un produit tensoriel est le produit des valeurs une-site
d = 3
E = np.eye(d)[:, :2]                     # plan E de C^3 (base orthonormee e1, e2)
PE = E @ E.conj().T                      # projecteur orthogonal sur E, rang 2
u = np.array([1.0, 1.0, 1.0]) / np.sqrt(3)
v = np.array([1.0, 0.0, -1.0]) / np.sqrt(2)
A1 = PE / np.sqrt(2)                     # ||A1||_F = 1 car tr(PE) = 2
A2 = np.outer(u, u.conj())
A3 = np.outer(v, v.conj())
C_prod = np.kron(np.kron(A1, A2), A3)    # produit tensoriel des trois blocs une-site

alpha = 0.3
q_mesure = q_alpha(C_prod, alpha, d)
q_attendu = q_un_site(A1, alpha) * q_un_site(A2, alpha) * q_un_site(A3, alpha)
print(f"q_alpha(A1 (x) A2 (x) A3) mesure  = {q_mesure:.12f}")
print(f"produit des q^(1)_alpha           = {q_attendu:.12f}   (ecart {abs(q_mesure - q_attendu):.2e})")

# --- Sanity 2 : les valeurs sur les blocs du minimiseur du Theoreme 1.2 (preuve de sharpness)
print(f"\nq^(1)_alpha(2^-1/2 P_E) = {q_un_site(A1, alpha):.12f}   papier : 1 - 2*alpha = {1 - 2 * alpha:.12f}")
print(f"q^(1)_alpha(|u><u|)     = {q_un_site(A2, alpha):.12f}   papier : 1 - alpha     = {1 - alpha:.12f}")

q_alpha(A1 (x) A2 (x) A3) mesure  = 0.196000000000
produit des q^(1)_alpha           = 0.196000000000   (ecart 5.55e-17)

q^(1)_alpha(2^-1/2 P_E) = 0.400000000000   papier : 1 - 2*alpha = 0.400000000000
q^(1)_alpha(|u><u|)     = 0.700000000000   papier : 1 - alpha     = 0.700000000000


### Lecture du résultat

Le moteur est en place et ses deux sanity checks passent à la précision flottante près :

- **Multiplicativité** : sur un produit tensoriel de blocs une-site, la forme à trois copies vaut exactement le produit des trois valeurs une-site — c'est l'identité combinatoire derrière la Proposition 2.1, mesurée plutôt qu'admise.
- **Les deux valeurs une-site du papier** : $q^{(1)}_\alpha(2^{-1/2}P_E) = 1 - 2\alpha$ et $q^{(1)}_\alpha(|u\rangle\langle u|) = 1-\alpha$ — ce sont les deux briques que la preuve de *sharpness* du Théorème 1.2 assemble (§3).

Le joueur du §4 le mesurera : le meilleur de 4000 tirages aléatoires de rang deux donne $q_{0{,}3} = 0{,}675526$, loin au-dessus de $0{,}196 = (1-2\alpha)(1-\alpha)^2$ : le minimum annoncé par le théorème n'est pas un tas de matrices ordinaires, il faut le construire.


## 3. La courbe exacte et son minimiseur (Théorème 1.2)

L'énoncé central du papier, pour des dimensions locales $2 \leq d_i \leq 4$ et $0 \leq \alpha \leq 1/2$ :

> $$\min\{\, q_\alpha(C) \;:\; \|C\| = 1,\ \operatorname{rang} C \leq 2 \,\} = (1-2\alpha)(1-\alpha)^2,$$
>
> les minimiseurs étant exactement $2^{-1/2}\, P_E \otimes |u\rangle\langle u| \otimes |v\rangle\langle v|$ à une phase et une permutation des sites près, avec $P_E$ le projecteur orthogonal sur un plan $E$ d'un espace local et $u, v$ des vecteurs unitaires des deux autres sites.

Trois conséquences immédiates. D'abord la valeur : à $\alpha < 1/2$ la courbe est $> 0$ — **aucune** matrice de rang deux n'a de forme négative, trois copies ne suffisent pas. Ensuite, à $\alpha > 1/2$, le même minimiseur donne $(1-2\alpha)(1-\alpha)^2 < 0$ — trois copies suffisent. Le seuil de distillabilité à trois copies est donc **exactement** $\alpha = 1/2$, le même qu'à une copie. Enfin la structure du minimiseur : un plan de Schmidt sur un site, des vecteurs purs sur les deux autres — ni symétrique, ni normal.

Le produit tensoriel construit au §2 comme sanity check est précisément ce minimiseur : $C^\star = 2^{-1/2}P_E \otimes |u\rangle\langle u| \otimes |v\rangle\langle v|$, de norme un et de rang deux. Vérifions ces trois propriétés, puis la courbe entière.

In [4]:
def minimiseur(d, plan=None, vec_u=None, vec_v=None, seed=7):
    """C* = 2^-1/2 P_E (x) |u><u| (x) |v><v| du Theoreme 1.2 : norme 1, rang 2.

    plan, vec_u, vec_v : colonnes orthonormales / vecteurs unitaires explicites.
    Par defaut, un plan et deux vecteurs aleatoires (seed fixe).
    """
    rng_loc = np.random.default_rng(seed)
    if plan is None:
        Q, _ = np.linalg.qr(rng_loc.normal(size=(d, 2)) + 1j * rng_loc.normal(size=(d, 2)))
        plan = Q
    if vec_u is None:
        x = rng_loc.normal(size=d) + 1j * rng_loc.normal(size=d)
        vec_u = x / np.linalg.norm(x)
    if vec_v is None:
        y = rng_loc.normal(size=d) + 1j * rng_loc.normal(size=d)
        vec_v = y / np.linalg.norm(y)
    PE = plan @ plan.conj().T
    return np.kron(np.kron(PE / np.sqrt(2), np.outer(vec_u, vec_u.conj())),
                   np.outer(vec_v, vec_v.conj()))


# Proprietes du minimiseur en d = 3
C_star = minimiseur(3, plan=E, vec_u=u, vec_v=v)
sing = np.linalg.svd(C_star, compute_uv=False)
print(f"||C*||_F          = {np.linalg.norm(C_star):.12f}   (attendu 1)")
print(f"valeurs singulieres non nulles = {sing[sing > 1e-12].round(12)}   (rang 2 : deux blocs egaux a 2^-1/2)")

# La courbe entiere : q_alpha(C*) contre (1 - 2 alpha)(1 - alpha)^2
alphas = np.linspace(0.0, 0.5, 11)
print("\nalpha     q_alpha(C*) mesure    courbe (1-2a)(1-a)^2     ecart")
ecart_max = 0.0
for a in alphas:
    qm = q_alpha(C_star, a, 3)
    qc = (1 - 2 * a) * (1 - a) ** 2
    ecart_max = max(ecart_max, abs(qm - qc))
    print(f"{a:.3f}      {qm: .12f}       {qc: .12f}        {abs(qm - qc):.1e}")
print(f"\nEcart maximal sur la grille : {ecart_max:.2e} — le minimiseur du theoreme atteint la courbe au flottant pres.")

||C*||_F          = 1.000000000000   (attendu 1)
valeurs singulieres non nulles = [0.70710678 0.70710678]   (rang 2 : deux blocs egaux a 2^-1/2)

alpha     q_alpha(C*) mesure    courbe (1-2a)(1-a)^2     ecart
0.000       1.000000000000        1.000000000000        0.0e+00
0.050       0.812250000000        0.812250000000        0.0e+00
0.100       0.648000000000        0.648000000000        0.0e+00
0.150       0.505750000000        0.505750000000        1.1e-16
0.200       0.384000000000        0.384000000000        5.6e-17
0.250       0.281250000000        0.281250000000        5.6e-17
0.300       0.196000000000        0.196000000000        5.6e-17
0.350       0.126750000000        0.126750000000        8.3e-17
0.400       0.072000000000        0.072000000000        2.2e-16
0.450       0.030250000000        0.030250000000        8.3e-17
0.500       0.000000000000        0.000000000000        1.1e-16

Ecart maximal sur la grille : 2.22e-16 — le minimiseur du theoreme atteint la courbe a

### Lecture du résultat

**Mesuré** : le minimiseur explicite du théorème — `C_star`, construit en trois lignes à partir d'un plan de dimension deux et de deux projecteurs de rang un — atteint la courbe $(1-2\alpha)(1-\alpha)^2$ au flottant près sur toute la grille, et c'est le **seul** comportement qu'on lui demande : le théorème 1.2 dit que c'est le **meilleur possible** parmi les matrices de norme un et de rang au plus deux.

Trois lectures de cette formule :

- **Le facteur $(1-2\alpha)$ s'annule exactement en $\alpha = 1/2$** : c'est le seuil de distillabilité à trois copies. Pour $\alpha > 1/2$, $q_\alpha(C_\star) < 0$ — et le critère du §2 transforme immédiatement ce nombre négatif en protocole de distillation.
- **Le facteur $(1-\alpha)^2$ ne s'annule jamais sur la bande** : les deux projecteurs $|u\rangle\langle u|$ et $|v\rangle\langle v|$ « coûtent » chacun $(1-\alpha)$ et aucun choix de $u$, $v$ ne peut faire mieux. Le seuil vit donc entièrement dans le facteur antisymétrique — celui que contrôle le plan $P_E$.
- **La courbe est positive sur toute la bande $1/d < \alpha \le 1/2$ en d = 3, 4** : trois copies de $\rho_\alpha$ n'y sont distillables par aucune matrice de rang deux. La bande de « bound entanglement NPT potentiel » résiste à trois copies dans ces dimensions.

Il reste à vérifier le mot **minimum** : rien n'interdit qu'une autre matrice de rang deux fasse mieux que $C_\star$. C'est l'objet du joueur suivant.

## 4. Joueur : chercher mieux que le théorème

Le théorème 1.2 affirme que **rien** de rang $\le 2$ ne descend sous la courbe. C'est une affirmation universelle — une infinité de matrices — qu'aucun échantillonnage ne peut prouver, mais qu'un échantillonnage peut **torturer** : si le théorème était faux, un tirage aléatoire bien fait finirait par trouver un contre-exemple.

L'espace à échantillonner est celui des matrices $d^3 \times d^3$ de norme de Frobenius un et de rang au plus deux. La paramétrisation la plus simple : $C = XY^\dagger$ avec $X, Y$ de $d^3 \times 2$, suivie d'une normalisation. Tirons-en un grand échantillon et mesurons le minimum de $q_\alpha$ atteint, contre la courbe.

In [5]:
def tirage_rang2(d, n_tirages, alpha, seed=123):
    """Echantillonne n_tirages matrices de norme 1 et rang <= 2, rend [q_alpha(C_j)] et le meilleur C."""
    rng_loc = np.random.default_rng(seed)
    meilleur_q = np.inf
    meilleur_C = None
    for _ in range(n_tirages):
        X = rng_loc.normal(size=(d**3, 2)) + 1j * rng_loc.normal(size=(d**3, 2))
        Y = rng_loc.normal(size=(d**3, 2)) + 1j * rng_loc.normal(size=(d**3, 2))
        C = X @ Y.conj().T
        C /= np.linalg.norm(C)
        q = q_alpha(C, alpha, d)
        if q < meilleur_q:
            meilleur_q, meilleur_C = q, C
    return meilleur_q, meilleur_C


# d = 3 : le meilleur de 4000 tirages par valeur de alpha, contre la courbe du theoreme
print("d = 3 — 4000 tirages de rang 2 par point, meilleurs q_alpha observes :\n")
print("alpha     meilleur tirage     courbe (1-2a)(1-a)^2    marge (tirage - courbe)")
for a in [0.2, 0.3, 0.35, 0.4, 0.45, 0.5]:
    q_t, _ = tirage_rang2(3, 4000, a)
    qc = (1 - 2 * a) * (1 - a) ** 2
    print(f"{a:.3f}      {q_t: .6f}           {qc: .6f}            {q_t - qc:+.4f}")
print("\nChaque marge est >= 0 et large (au moins 0.39 ici) : aucun tirage ne descend")
print("sous la courbe, et le tirage uniforme ne l'effleure meme pas — le minimiseur")
print("vit sur une famille de dimension finie (choix du plan, de u, de v) que le")
print("hasard balaye tres mal.")

d = 3 — 4000 tirages de rang 2 par point, meilleurs q_alpha observes :

alpha     meilleur tirage     courbe (1-2a)(1-a)^2    marge (tirage - courbe)


0.200       0.774437            0.384000            +0.3904


0.300       0.675526            0.196000            +0.4795


0.350       0.626995            0.126750            +0.5002


0.400       0.579719            0.072000            +0.5077


0.450       0.533944            0.030250            +0.5037


0.500       0.489669            0.000000            +0.4897

Chaque marge est >= 0 et large (au moins 0.39 ici) : aucun tirage ne descend
sous la courbe, et le tirage uniforme ne l'effleure meme pas — le minimiseur
vit sur une famille de dimension finie (choix du plan, de u, de v) que le
hasard balaye tres mal.


Même joueur en dimension quatre — la deuxième dimension couverte par le théorème. La matrice à trois sites y est de taille $64 \times 64$, chaque tirage coûte donc un peu plus cher ; l'échantillon est réduit en conséquence, mais il reste largement assez grand pour qu'un contre-exemple éventuel ait eu sa chance.

In [6]:
# d = 4 : meme joueur, 2000 tirages par point (matrices 64 x 64)
import time
t0 = time.perf_counter()
print("d = 4 — 2000 tirages de rang 2 par point, meilleurs q_alpha observes :\n")
print("alpha     meilleur tirage     courbe (1-2a)(1-a)^2    marge (tirage - courbe)")
for a in [0.25, 0.35, 0.45, 0.5]:
    q_t, _ = tirage_rang2(4, 2000, a, seed=456)
    qc = (1 - 2 * a) * (1 - a) ** 2
    print(f"{a:.3f}      {q_t: .6f}           {qc: .6f}            {q_t - qc:+.4f}")
print(f"\n({time.perf_counter() - t0:.1f} s pour l'ensemble)")
print("Meme constat qu'en d = 3 : la courbe du theoreme est la frontiere observee du nuage.")

d = 4 — 2000 tirages de rang 2 par point, meilleurs q_alpha observes :

alpha     meilleur tirage     courbe (1-2a)(1-a)^2    marge (tirage - courbe)


0.250       0.805213            0.281250            +0.5240


0.350       0.734569            0.126750            +0.6078


0.450       0.667471            0.030250            +0.6372


0.500       0.635130            0.000000            +0.6351

(1.3 s pour l'ensemble)
Meme constat qu'en d = 3 : la courbe du theoreme est la frontiere observee du nuage.


### Lecture du résultat

**Mesuré** : sur plusieurs milliers de matrices de rang deux tirées au hasard, en d = 3 comme en d = 4, le minimum observé de $q_\alpha$ reste **au-dessus** de la courbe $(1-2\alpha)(1-\alpha)^2$ — et l'écart est large partout : au moins $0{,}39$ en d = 3, au moins $0{,}52$ en d = 4. Il n'est pas monotone en $\alpha$ : en d = 3 il croît jusqu'à $\alpha = 0{,}40$ ($+0{,}5077$) puis redescend au dernier point ($+0{,}4897$ en $0{,}50$) ; en d = 4 il croît sur la grille puis amorce la même retombée au dernier point ($+0{,}6372$ en $0{,}45$, $+0{,}6351$ en $0{,}50$). Le tirage typique descend quand $\alpha$ augmente, et la courbe descend encore plus vite sur l'essentiel de la plage — assez pour garder la marge large partout, pas pour la faire croître jusqu'au bord.

Deux points de lecture :

- **Ce joueur ne prouve rien** — un échantillon ne peut pas réfuter « pour tout $C$ » — mais il rend le théorème *vivable* : la borne n'est pas simplement respectée, elle est infranchissable de loin. Un théorème faux avec une constante non optimale aurait été trahi par une marge franchement négative quelque part ; ici le signe même de la marge ne vacille jamais.
- **Pourquoi l'écart reste-t-il si large ?** Le minimiseur $C_\star$ n'est pas une matrice générique : il vit sur la famille finie-dimensionnelle $2^{-1/2}P_E \otimes |u\rangle\langle u| \otimes |v\rangle\langle v|$ (choix du plan $E$, des vecteurs $u$, $v$, d'une phase) — une sous-variété de mesure nulle dans l'espace des matrices de rang deux. Le tirage uniforme ne peut pas l'approcher : il confirme la **frontière**, il ne la **touche** pas. C'est le comportement attendu d'un minimum dégénéré, et la raison pour laquelle le papier doit prouver par décomposition en blocs de symétrie (section III) ce que l'échantillonnage ne verra jamais.


## 5. Toute dimension : la borne 37/80 et la frontière ouverte

Le théorème 1.2 s'arrête à $d \le 4$. Le second résultat du papier va dans l'autre direction — il couvre **toutes** les dimensions finies, au prix d'une borne moins fine :

> **Théorème 1.3 (cité, section IV du papier).** *En toute dimension locale $d \ge 3$, trois copies de $\rho_\alpha$ ne sont pas distillables dès que $\alpha \le 37/80$.*

Ce que cette borne garantit, et ce qu'elle ne dit pas :

- **Garanti (mesurable)** : pour tout $d \ge 3$, l'intervalle ouvert $\left(\frac{1}{d},\ \frac{37}{80}\right]$ contient des valeurs de $\alpha$ — donc il existe des états de Werner **NPT** ($\alpha > 1/d$) qui **survivent à trois copies** dans toute dimension. La question « undistillability implies PPT ? » a une réponse négative pour le protocole à trois copies, en toute dimension : trois copies ne suffisent pas à distinguer NPT-distillable de NPT-undistillable.
- **Non garanti** : la forme exacte du minimum pour $d \ge 5$. Entre $\frac{37}{80} = 0{,}4625$ et $\frac{1}{2}$, la porte reste ouverte : le papier conjecture que la courbe $(1-2\alpha)(1-\alpha)^2$ persiste en toute dimension, mais ne le démontre pas.

La méthode du papier pour ce théorème est d'une nature différente : là où le théorème 1.2 décompose par blocs de symétrie, le théorème 1.3 exhibe un **certificat rationnel dimension-uniforme** — une majoration explicite de $q_\alpha$ sur les matrices de rang deux dont les coefficients sont indépendants de $d$. C'est ce qui lui permet de couvrir toutes les dimensions d'un coup.

Voyons ce que nos joueurs numériques en disent en $d = 5$ — la première dimension **au-delà** du théorème 1.2, où seule la borne $37/80$ est acquise.

In [7]:
# d = 5 : dimension non couverte par le theoreme 1.2. Quatre reperes par alpha :
#   - le meilleur tirage aleatoire de rang 2,
#   - le minimiseur produit C* (conjecture : (1-2a)(1-a)^2),
#   - la zone : borne due (thm 1.3, a <= 37/80), zone ouverte (conjecture), ou distillable (a > 1/2).
def zone(a):
    if a <= 37 / 80:
        return "borne due (thm 1.3)"
    if a <= 0.5:
        return "ZONE OUVERTE"
    return "distillable (C* < 0)"


print("d = 5 — 800 tirages + minimiseur produit, par point :\n")
print("alpha     meilleur tirage    q_alpha(C*)      (1-2a)(1-a)^2    zone")
for a in [0.3, 0.4, 0.4625, 0.48, 0.5, 0.55]:
    q_t, _ = tirage_rang2(5, 800, a, seed=789)
    C5 = minimiseur(5)
    q_prod = q_alpha(C5, a, 5)
    qc = (1 - 2 * a) * (1 - a) ** 2
    print(f"{a:.4f}      {q_t: .6f}         {q_prod: .6f}        {qc: .6f}      {zone(a)}")
print("\nDeux leçons : (1) dans la zone ouverte 37/80 < a <= 1/2, le numerique ne")
print("contredit pas la conjecture (aucun tirage sous la courbe), mais aucun")
print("theoreme ne la couvre ; (2) des a > 1/2, C* donne q < 0 en toute dimension —")
print("la distillabilite y est un certificat explicite, pas une conjecture.")

d = 5 — 800 tirages + minimiseur produit, par point :

alpha     meilleur tirage    q_alpha(C*)      (1-2a)(1-a)^2    zone


0.3000       0.819275          0.196000         0.196000      borne due (thm 1.3)


0.4000       0.763452          0.072000         0.072000      borne due (thm 1.3)


0.4625       0.729615          0.021668         0.021668      borne due (thm 1.3)


0.4800       0.720282          0.010816         0.010816      ZONE OUVERTE


0.5000       0.709690          0.000000         0.000000      ZONE OUVERTE


0.5500       0.683449         -0.020250        -0.020250      distillable (C* < 0)

Deux leçons : (1) dans la zone ouverte 37/80 < a <= 1/2, le numerique ne
contredit pas la conjecture (aucun tirage sous la courbe), mais aucun
theoreme ne la couvre ; (2) des a > 1/2, C* donne q < 0 en toute dimension —
la distillabilite y est un certificat explicite, pas une conjecture.


### Lecture du résultat

**Mesuré** : en $d = 5$, le minimiseur produit $C_\star$ — celui dont le théorème 1.2 garantit l'optimalité en $d \le 4$ — donne toujours $q_\alpha(C_\star) = (1-2\alpha)(1-\alpha)^2$ au flottant près (c'est l'identité du §3, valable en toute dimension : le calcul est purement local). Et le tirage aléatoire reste au-dessus.

**Mais l'interprétation change de statut** : en $d \le 4$, « le tirage ne descend pas sous la courbe » *confirme* un théorème ; en $d = 5$, il *nourrit* une conjecture. La colonne de droite dit où est la frontière du connu :

| Zone de $\alpha$ | $d = 3, 4$ | $d \ge 5$ |
|---|---|---|
| $\alpha \le 1/d$ | séparable (PPT, une copie suffit) | séparable (PPT) |
| $1/d < \alpha \le 37/80$ | **trois copies ne distillent pas** (thm 1.2) | **trois copies ne distillent pas** (thm 1.3) |
| $37/80 < \alpha \le 1/2$ | **trois copies ne distillent pas** (thm 1.2) | **ouvert** — conjecture : ne distillent pas |
| $\alpha > 1/2$ | distillables à trois copies (thm 1.2 : le critère s'inverse) | distillables à trois copies (le même $C_\star$ fonctionne : c'est une inégalité, pas une égalité) |

La dernière ligne mérite un mot : la partie « $\alpha > 1/2 \Rightarrow$ distillable » ne dépend **pas** de la dimension. $C_\star$ y donne $q_\alpha < 0$ en toute dimension (le calcul du §3, encore), et $q_\alpha(C) < 0$ est *toujours* un certificat de distillabilité — c'est la direction facile du critère du §2. Seule la direction « rien d'autre ne fait mieux » (l'optimalité) est ouverte en $d \ge 5$.

## 6. Ce que le résultat dit — et ce qu'il ne dit pas

Replaçons-le dans le paysage d'InfoQuant-01, section « la distillabilité et son trou ». La question d'origine — *undistillability implies PPT ?* — demande s'il existe un état NPT dont **aucun nombre** de copies n'est distillable. Ce carnet a examiné le premier étage de l'escalade : **trois** copies.

**Ce que le résultat OQP 2 établit** (cité, depuis la copie archivée) :

1. En $d = 3, 4$, trois copies de $\rho_\alpha$ sont distillables **si et seulement si** $\alpha > 1/2$ — un seuil **strictement au-dessus** du seuil NPT ($1/d$) et du seuil de distillabilité à une copie ($\lambda > 3(d-1)/(4d-2)$, soit encore $\alpha > 1/2$ — le DürCritère d'InfoQuant-01). La bande $1/d < \alpha \le 1/2$ est donc peuplée d'états **NPT mais non distillables à trois copies**.
2. En toute dimension $d \ge 3$, la même non-distillabilité tient jusqu'à $\alpha \le 37/80$ — des états de Werner « three-copy-bound-entangled » existent partout.
3. Le seuil $\alpha = 1/2$ coïncide avec le seuil une-copie de Dür : pour les Werner, passer d'une à trois copies **ne gagne rien**. C'est un indice structurel fort : la distillabilité des Werner ne s'améliore peut-être jamais avec le nombre de copies — exactement le comportement qu'exigerait un contre-exemple à la conjecture.

**Ce qu'il ne dit PAS** :

- Rien sur $n \ge 4$ copies. La bande $1/d < \alpha \le 1/2$ reste non distillable à trois copies, mais quatre, cinq, $n$ copies ? Ouvert. Le mot « trois » dans le titre du papier est aussi une limite de portée.
- Rien sur les états **non invariants par échange**. Les Werner sont une famille à un paramètre, très symétrique ; la conjecture générale porte sur *tous* les états NPT.
- Rien sur la clôture PPT des opérations localement inversibles, ni sur la question une-copie (un état NPT à une copie est-il distillable à un nombre *fini* de copies ?) — le papier s'inscrit dans le programme mais ne le referme pas.

En résumé : la conjecture « NPT $\Rightarrow$ distillable » reste ouverte ; ce résultat renforce l'opinion (défendue par plusieurs écoles) qu'elle est **fausse**, en exhibant la première famille d'états dont la non-distillabilité résiste à trois copies en toute dimension — avec, en $d = 3, 4$, la courbe exacte qui dit précisément jusqu'où trois copies peuvent aller.

## 7. Exercices

Trois exercices pour s'approprier le critère. Le premier fait manipuler la forme $q_\alpha$ sur une famille explicite ; le deuxième étend le joueur du §4 à une recherche guidée ; le troisième demande de vérifier numériquement le pont de paramétrisation du §1. Chaque cellule s'exécute telle quelle (squelettes complets, `result = None  # TODO etudiant`) — à vous de remplir.

### Exercice 1 — Deux copies ne suffisent pas non plus

La forme à **deux** copies s'obtient en appliquant la définition générale du §2 à deux facteurs : $q^{(2)}_\alpha(D) = \|D\|^2 - \alpha\|\mathrm{Tr}_R D\|^2 - \alpha\|\mathrm{Tr}_L D\|^2 + \alpha^2 |\mathrm{Tr}\, D|^2$ pour $D$ opérateur sur $\mathbb{C}^d \otimes \mathbb{C}^d$ — en $\alpha = 1/2$, c'est la forme deux sites $Q_{L|R}$ de l'équation 5 du papier. **Question** : montrer numériquement que pour $d = 3$, le minimum de $q^{(2)}_\alpha$ sur les $D$ de norme un et de rang deux est **positif pour tout $\alpha \le 1/2$** — deux copies ne distillent pas — mais **négatif dès $\alpha > 1/2$**. Indice : écrire `q2_alpha` sur le modèle de `q_alpha`, avec trois termes de trace seulement (gauche, droite, totale) — `tr_sites` s'applique tel quel avec deux sites. *(La question deux-copies a une histoire récente : références [14], [16], [17] du papier, une vague de préprints 2026 — voir la section Références.)*

In [8]:
# Exercice 1 a completer : forme a deux copies q2_alpha et son minimum sur le rang deux.
# Etape 1 : ecrire q2_alpha(D, alpha, d) = ||D||^2 - a*||Tr_R D||^2 - a*||Tr_L D||^2 + a^2*|Tr D|^2.
#   (Tr_R et Tr_L : traces partielles sur le second / le premier site -- reutiliser tr_sites,
#   ici avec d = dimension d'un site et C vivant dans (C^d x C^d) : reshape en (d, d, d, d).)
# Etape 2 : echantillonner des D de rang 2 : D = X @ Y.conj().T avec X, Y de (d^2, 2), normalises.
# Etape 3 : mesurer le meilleur q2 observe pour alpha dans [0.2, 0.6] ; ou s'annule-t-il ?

def q2_alpha(D, alpha, d):
    """Forme a deux copies sur un operateur D de (C^d x C^d)."""
    # TODO etudiant
    return None


result = None  # TODO etudiant : meilleure q2 observee par alpha, et seuil numerique
if result is None:
    print("Exercice 1 a completer")

Exercice 1 a completer


### Exercice 2 — Recherche guidée : descente de gradient sur $q_\alpha$

Le joueur du §4 tire **au hasard** et constate qu'il ne descend pas sous la courbe. Faites mieux : une descente de gradient sur les facteurs $X, Y$ de $C = XY^\dagger / \|XY^\dagger\|$, en utilisant le gradient de $q_\alpha$ par différences finies (ou, pour les plus ambitieux, autodiff de `q_alpha` à la main). **Question** : partant de $C_\star$ légèrement bruité, la descente converge-t-elle vers $C_\star$ ? Partant d'un point aléatoire, tombe-t-on sous la courbe ? Indice : la fonction `q_alpha` est quadratique en $C$ — son gradient est linéaire, la descente devrait être bien comportée ; c'est la contrainte de rang deux qui rend le problème intéressant.

In [9]:
# Exercice 2 a completer : descente de gradient sur les facteurs du rang 2.
# Etape 1 : definir C(X, Y) = (X @ Y.conj().T) normalise, puis f(X, Y) = q_alpha(C(X, Y), alpha, d).
# Etape 2 : gradient par differences finies sur chaque entree complexe de X et Y
#   (ou parametrisation reelle : parties reelle et imaginaire separees).
# Etape 3 : 200 pas de descente depuis (a) C_star bruite, (b) un tirage aleatoire ;
#   comparer la valeur finale a la courbe (1-2a)(1-a)^2.

def descente_q(alpha, d, n_pas=200, pas=0.02, depart=None, seed=99):
    """Descente sur (X, Y) ; rend la trajectoire des q_alpha(C_j)."""
    # TODO etudiant
    return None


result = None  # TODO etudiant : trajectoires depuis C_star bruite et depuis un hasard
if result is None:
    print("Exercice 2 a completer")

Exercice 2 a completer


### Exercice 3 — Le pont λ↔α sous contrainte

Le §1 a converti la paramétrisation $\lambda$ d'InfoQuant-01 (Horodecki) en $\alpha$ du papier. **Question** : le seuil une-copie de Dür, $\lambda > 3(d-1)/(4d-2)$, se traduit en $\alpha > 1/2$ **exactement, pour toute dimension**. Le vérifier numériquement pour $d = 3, \dots, 12$ par substitution directe : composer `alpha_de_lambda` avec `lambda_seuil_dür(d)` et montrer que l'écart à $1/2$ est au flottant près. **Bonus** : montrer algébriquement (sur le papier, puis le constater en symboles avec sympy) que la composition se simplifie en $1/2$ indépendamment de $d$.

In [10]:
# Exercice 3 a completer : le seuil une-copie de Dur en parametrisation alpha, pour toute dimension.
# Etape 1 : lambda_dur(d) = 3*(d-1)/(4*d-2) ; composer avec alpha_de_lambda.
# Etape 2 : boucle sur d = 3..12, mesurer alpha_de_lambda(lambda_dur(d)) - 0.5 (attendu ~1e-16).
# Etape 3 (bonus) : refaire le calcul en exact avec sympy et simplifier.

result = None  # TODO etudiant : tableau (d, ecart au seuil 1/2) + simplification symbolique
if result is None:
    print("Exercice 3 a completer")

Exercice 3 a completer


## À retenir

- **La question** : *undistillability implies PPT ?* — existe-t-il un état NPT qu'aucun nombre de copies ne distille ? Le test le plus bas de l'escalade : **trois** copies de l'état de Werner $\rho_\alpha$.
- **Le critère de travail** (Proposition 2.1) : $\rho_\alpha^{\otimes 3}$ est distillable si et seulement s'il existe $C$ de norme un et de rang $\le 2$ avec $q_\alpha(C) = \sum_S (-\alpha)^{|S|} \|\mathrm{Tr}_S C\|^2 < 0$. Une **inégalité numérique**, calculable en trois lignes de numpy.
- **Le résultat** (MacCormack–Bao, 23/10/2026, preprint non relu) : en $d = 3, 4$, ce minimum vaut exactement $(1-2\alpha)(1-\alpha)^2$ — donc trois copies distillent **ssi $\alpha > 1/2$** ; et en toute dimension $d \ge 3$, trois copies ne distillent pas pour $\alpha \le 37/80$. Des Werner NPT survivant à trois copies existent en toute dimension.
- **Le minimiseur** est un produit tensoriel simple : $C_\star = 2^{-1/2} P_E \otimes |u\rangle\langle u| \otimes |v\rangle\langle v|$, chaque facteur portant sa valeur une-site — $(1-2\alpha)$ pour le plan antisymétrique, $(1-\alpha)$ pour chaque projecteur de rang un.
- **La coïncidence structurante** : le seuil trois-copies $\alpha = 1/2$ est **aussi** le seuil une-copie de Dür et al. Pour les Werner, l'escalade en nombre de copies ne gagne rien de une à trois — un indice que la conjecture « NPT $\Rightarrow$ distillable » pourrait être fausse.
- **Statut** : preprint non relu ; mathématiques développées par les chercheurs de Physical Superintelligence avec assistance de modèles IA, formalisation Lean du cas qutrit publiée séparément (48 statements, pin `v4.35.0-rc4`) — le présent carnet n'en dépend pas (pas de port Lake, cf. l'arbitrage #19898 dans la PR).

## Références

- **Ian MacCormack, Ning Bao**, *Three copies of the Werner state: the sharp curve in dimensions three and four and a bound in every dimension*, preprint du 23 octobre 2026, 14 pages, dossier `oqp02-three-copy-werner` du lot `physical-superintelligence/oqp-batch-2026`. **Copie archivée** : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Physics\2026 - MacCormack, Bao (PSI) - Three copies of the Werner state - sharp curve d=3-4 and bound in every dimension (OQP 02 three-copy-werner, preprint).pdf` — tous les énoncés cités dans ce carnet (Théorèmes 1.2, 1.3, Proposition 2.1, équations 1, 2, 5, 6) sont lus depuis cette copie.
- **Le critère de Schmidt rang deux** et la définition de la distillabilité à $n$ copies : le papier les attribue à **[3]** M. Horodecki, P. Horodecki, R. Horodecki, *Mixed-state entanglement and distillation: is there a "bound" entanglement in nature?*, Phys. Rev. Lett. 80, 5239 (1998) ; **[4]** D. P. DiVincenzo, P. W. Shor, J. A. Smolin, B. M. Terhal, A. V. Thapliyal, *Evidence for bound entangled states with negative partial transpose*, Phys. Rev. A 61, 062312 (2000) ; **[5]** W. Dür, J. I. Cirac, M. Lewenstein, D. Bruß, *Distillability and partial transposition in bipartite systems*, Phys. Rev. A 61, 062313 (2000) — références [3]–[5] du papier.
- **Le critère PPT** : **[21]** A. Peres, *Separability criterion for density matrices*, Phys. Rev. Lett. 77, 1413 (1996) ; **[19]** M. Horodecki, P. Horodecki, R. Horodecki, *Separability of mixed states: necessary and sufficient conditions*, Phys. Lett. A 223, 1 (1996).
- **L'état de Werner** : **[6]** R. F. Werner, *Quantum states with Einstein-Podolsky-Rosen correlations admitting a hidden-variable model*, Phys. Rev. A 40, 4277 (1989).
- **La vague deux-copies 2026** (contexte de l'exercice 1) : **[14]** J. Fu, L. Gao, S.-J. Park, *A solution to 2-copy distillability of Werner states*, arXiv:2607.21367 ; **[16]** T. C. Fraser, F. Huber, B. Pozsgay, I. Vona, *On the two-copy distillability of Werner states and a new partial trace inequality*, arXiv:2607.24309 ; **[17]** K. Bharti, R. Gajjala, T. Haug, *Two-copy nondistillability of Werner states: sharp partial-trace inequalities and finite-copy extensions*, arXiv:2607.24479 — références [14], [16], [17] du papier.
- **Certificats exacts publiés** (reçus et SHA256SUMS dans le package `verification/` du dépôt amont, référence [20] du papier) : **[1]** certificat qutrit (Zenodo, doi:10.5281/zenodo.23134454) ; **[2]** A.-X. Liu, certificat two-ququart (`github.com/GLiuax/d4-three-copy-werner`, commit 2ba792f).
- **Carnet précédent de la sous-série** : *InfoQuant-01 — Intrication des états de Werner : PPT et distillabilité* (`InfoQuant-01-Intrication-Werner-PPT-Distillation-Python.ipynb`) — paramétrisation λ (Horodecki), critère de réduction PPT, seuil une-copie de Dür et al., le « trou » de distillabilité que le présent carnet commence à combler.
- **Dépôt amont** : `physical-superintelligence/oqp-batch-2026`, dossier `oqp02-three-copy-werner/` — README (provenance IA déclarée), `Challenge.lean` (formalisation du cas qutrit, 48 statements, pin Mathlib `v4.35.0-rc4`), package `verification/`. Non porté dans ce dépôt (arbitrage « carnet sans lake » #19898, même décision que pour les grains 27/12/38).

*Provenance et statut : preprint de recherche du 23 octobre 2026, non relu ; les auteurs déclarent une assistance de modèles IA pour le développement mathématique (chercheurs PSI) et la formalisation Lean (GPT-6-Astra), MacCormack et Bao portant la responsabilité de la correction. Ce carnet vérifie ce qui est numériquement vérifiable depuis la copie archivée et cite le reste.*